In [ ]:
# Cell 1: Mount Drive + Verify Dataset2 structure
from google.colab import drive
drive.mount('/content/drive')

import os
from glob import glob

base_path = '/content/drive/MyDrive/MHCNNFD_Data/'
folders = [
    'Evening_Fire_Incident_aug_img',
    'Evening_Forest_Condition_aug_img',
    'Pre-_Evening_Fire_Incident_aug_img',
    'Pre-_Evening_Forest_Condition_aug_img'
]

print("Dataset2 (MHCNNFD_Data) verification:")
print("=" * 60)

total_images = 0
for folder in folders:
    folder_path = os.path.join(base_path, folder)
    if os.path.exists(folder_path):
        png_count = len(glob(os.path.join(folder_path, '*.png')))
        jpg_count = len(glob(os.path.join(folder_path, '*.jpg')))
        total = png_count + jpg_count
        total_images += total
        print(f"{folder}: {total:,} images ({png_count} PNG, {jpg_count} JPG)")
    else:
        print(f"{folder}: NOT FOUND")

print(f"\nTOTAL Dataset2 images: {total_images:,}")
print(f"Expected ~15,560: {'✓' if 15000 <= total_images <= 16000 else '✗'}")


Mounted at /content/drive
Dataset2 (MHCNNFD_Data) verification:
Evening_Fire_Incident_aug_img: 3,890 images (3890 PNG, 0 JPG)
Evening_Forest_Condition_aug_img: 3,890 images (3890 PNG, 0 JPG)
Pre-_Evening_Fire_Incident_aug_img: 3,890 images (3890 PNG, 0 JPG)
Pre-_Evening_Forest_Condition_aug_img: 3,890 images (3890 PNG, 0 JPG)

TOTAL Dataset2 images: 15,560
Expected ~15,560: ✓


In [ ]:
import cv2
import numpy as np
import os
from glob import glob
import pandas as pd
from sklearn.model_selection import train_test_split
import random
random.seed(42)  # Reproducible split

# Paths
source_base = '/content/drive/MyDrive/MHCNNFD_Data/'
stress_base = '/content/drive/MyDrive/MHCNNFD_StressTest/'
source_folders = [
    'Evening_Fire_Incident_aug_img',
    'Evening_Forest_Condition_aug_img',
    'Pre-_Evening_Fire_Incident_aug_img',
    'Pre-_Evening_Forest_Condition_aug_img'
]

# Recreate exact original test set (1,556 images)
print("Recreating original test split...")
all_image_paths = []
all_labels = []

for folder in source_folders:
    class_dir = os.path.join(source_base, folder)
    paths = glob(os.path.join(class_dir, '*.png'))
    all_image_paths.extend(paths)
    all_labels.extend([folder] * len(paths))

df = pd.DataFrame({'path': all_image_paths, 'label': all_labels})
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp['label'])

print(f"Original test set: {len(df_test)} images ({len(df_test)/4:.0f} per class)")

# Create 3 stress test folders
stress_types = ['test_gaussian', 'test_blur', 'test_fog']
for stress_name in stress_types:
    stress_dir = os.path.join(stress_base, stress_name)
    for folder in source_folders:
        os.makedirs(os.path.join(stress_dir, folder), exist_ok=True)

# Process each test image through 3 stress types
print("\nCreating stressed test sets (originals untouched)...")
for idx, row in df_test.iterrows():
    img_path = row['path']
    folder_name = row['label']
    img = cv2.imread(img_path)

    # Version A: Gaussian Noise (cheap drone camera)
    noise = np.random.normal(0, 25, img.shape).astype(np.uint8)
    gaussian_img = cv2.add(img, noise)
    cv2.imwrite(os.path.join(stress_base, 'test_gaussian', folder_name, os.path.basename(img_path)), gaussian_img)

    # Version B: Motion Blur (shaky drone)
    blur_img = cv2.GaussianBlur(img, (9, 9), 0)
    cv2.imwrite(os.path.join(stress_base, 'test_blur', folder_name, os.path.basename(img_path)), blur_img)

    # Version C: Fog/Haze (smoke)
    overlay = np.full(img.shape, 180, dtype=np.uint8)
    fog_img = cv2.addWeighted(img, 0.3, overlay, 0.7, 0)
    cv2.imwrite(os.path.join(stress_base, 'test_fog', folder_name, os.path.basename(img_path)), fog_img)

print("\n✓ CREATED 3 STRESSED TEST SETS:")
print(f"  test_gaussian/: 1,556 noisy images")
print(f"  test_blur/:    1,556 blurred images")
print(f"  test_fog/:     1,556 foggy images")
print("\nOriginal Dataset2: UNTOUCHED")


Recreating original test split...
Original test set: 1556 images (389 per class)

Creating stressed test sets (originals untouched)...

✓ CREATED 3 STRESSED TEST SETS:
  test_gaussian/: 1,556 noisy images
  test_blur/:    1,556 blurred images
  test_fog/:     1,556 foggy images

Original Dataset2: UNTOUCHED


In [ ]:
import os
from glob import glob

base_path = '/content/drive/MyDrive/MHCNNFD_StressTest/'
class_folders = [
    'Evening_Fire_Incident_aug_img',
    'Evening_Forest_Condition_aug_img',
    'Pre-_Evening_Fire_Incident_aug_img',
    'Pre-_Evening_Forest_Condition_aug_img'
]

print("🔍 COMPLETE FOLDER VERIFICATION 🔍")
print("="*60)

# Check all stress test folders
for stress_type in ['test_gaussian', 'test_blur', 'test_fog']:
    stress_path = os.path.join(base_path, stress_type)
    print(f"\n {stress_type.upper()}:")

    total = 0
    for class_folder in class_folders:
        folder_path = os.path.join(stress_path, class_folder)
        if os.path.exists(folder_path):
            img_count = len(glob(os.path.join(folder_path, '*.png')))
            total += img_count
            print(f"  {class_folder}: {img_count:3d} images")
        else:
            print(f"  {class_folder}: ❌ MISSING")

    print(f"  TOTAL: {total}/1556 ✓" + (" PASS" if total == 1556 else " FAIL"))

print("\n VERIFICATION COMPLETE")
print("Next: Ready for model evaluation on stress tests!")


🔍 COMPLETE FOLDER VERIFICATION 🔍

 TEST_GAUSSIAN:
  Evening_Fire_Incident_aug_img: 389 images
  Evening_Forest_Condition_aug_img: 389 images
  Pre-_Evening_Fire_Incident_aug_img: 389 images
  Pre-_Evening_Forest_Condition_aug_img: 389 images
  TOTAL: 1556/1556 ✓ PASS

 TEST_BLUR:
  Evening_Fire_Incident_aug_img: 389 images
  Evening_Forest_Condition_aug_img: 389 images
  Pre-_Evening_Fire_Incident_aug_img: 389 images
  Pre-_Evening_Forest_Condition_aug_img: 389 images
  TOTAL: 1556/1556 ✓ PASS

 TEST_FOG:
  Evening_Fire_Incident_aug_img: 389 images
  Evening_Forest_Condition_aug_img: 389 images
  Pre-_Evening_Fire_Incident_aug_img: 389 images
  Pre-_Evening_Forest_Condition_aug_img: 389 images
  TOTAL: 1556/1556 ✓ PASS

 VERIFICATION COMPLETE
Next: Ready for model evaluation on stress tests!


In [ ]:
# === STEP 1: MOUNT + SETUP ===
from google.colab import drive
drive.mount('/content/drive')
import cv2, numpy as np, os, pandas as pd
from sklearn.model_selection import train_test_split
from glob import glob

# Paths (EXACT from StressTestCreation.ipynb)
sourcebase = '/content/drive/MyDrive/MHCNNFDData'
stressbase = '/content/drive/MyDrive/MHCNNFDStressTest'
sourcefolders = ['EveningFireIncidentaugimg','EveningForestConditionaugimg',
                'Pre-EveningFireIncidentaugimg','Pre-EveningForestConditionaugimg']

print("✅ Paths ready. Creating testlightgauss...")

# === STEP 2: RECREATE TEST SPLIT (1,556 images) ===
allimagepaths, alllabels = [], []
for folder in sourcefolders:
    classdir = os.path.join(sourcebase, folder)
    paths = glob(os.path.join(classdir, '*.png'))
    allimagepaths.extend(paths)
    alllabels.extend([folder] * len(paths))

df = pd.DataFrame({'path': allimagepaths, 'label': alllabels})
dftrain, dftemp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
dfval, dftest = train_test_split(dftemp, test_size=0.5, random_state=42, stratify=dftemp['label'])
print(f"✅ Test set ready: {len(dftest)} images")

# === STEP 3: CREATE NEW FOLDER testlightgauss (std=10) ===
lightgauss_dir = os.path.join(stressbase, 'testlightgauss')
for folder in sourcefolders:
    os.makedirs(os.path.join(lightgauss_dir, folder), exist_ok=True)

print("🎯 Creating LIGHT GAUSSIAN (std=10)...")
for idx, row in dftest.iterrows():
    imgpath = row['path']
    foldername = row['label']
    img = cv2.imread(imgpath)

    # LIGHT Gaussian std=10 (realistic drone noise)
    noise = np.random.normal(0, 10, img.shape).astype(np.uint8)
    light_img = cv2.add(img, noise)

    # Save to NEW folder
    outpath = os.path.join(lightgauss_dir, foldername, os.path.basename(imgpath))
    cv2.imwrite(outpath, light_img)

print("✅ SAVED: testlightgauss/ (1,556 images, std=10)")
print("📁 Full path:", lightgauss_dir)


Mounted at /content/drive
✅ Paths ready. Creating testlightgauss...


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from glob import glob

# CHECK exact path structure
basepath = '/content/drive/MyDrive/MHCNNFDData'
folders = ['EveningFireIncidentaugimg','EveningForestConditionaugimg',
           'Pre-EveningFireIncidentaugimg','Pre-EveningForestConditionaugimg']

print("🔍 DEBUGGING PATHS:")
for folder in folders:
    folderpath = os.path.join(basepath, folder)
    pngs = glob(os.path.join(folderpath, '*.png'))
    print(f"{folder}: {len(pngs)} images → {folderpath}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 DEBUGGING PATHS:
EveningFireIncidentaugimg: 0 images → /content/drive/MyDrive/MHCNNFDData/EveningFireIncidentaugimg
EveningForestConditionaugimg: 0 images → /content/drive/MyDrive/MHCNNFDData/EveningForestConditionaugimg
Pre-EveningFireIncidentaugimg: 0 images → /content/drive/MyDrive/MHCNNFDData/Pre-EveningFireIncidentaugimg
Pre-EveningForestConditionaugimg: 0 images → /content/drive/MyDrive/MHCNNFDData/Pre-EveningForestConditionaugimg


In [ ]:
# === MOUNT + YOUR EXACT FOLDERS ===
from google.colab import drive
drive.mount('/content/drive')
import cv2, numpy as np, os, pandas as pd
from sklearn.model_selection import train_test_split
from glob import glob

sourcebase = '/content/drive/MyDrive/MHCNNFDData'
stressbase = '/content/drive/MyDrive/MHCNNFDStressTest'

# ✅ YOUR EXACT EXTRACTED FOLDER NAMES
sourcefolders = [
    'Evening_Fire_Incident_aug_img',
    'Evening_Forest_Condition_aug_img',
    'Pre-_Evening_Fire_Incident_aug_img',
    'Pre-_Evening_Forest_Condition_aug_img'
]

print("🔍 Verifying YOUR folders:")
total = 0
for folder in sourcefolders:
    folderpath = os.path.join(sourcebase, folder)
    pngs = glob(os.path.join(folderpath, '*.png'))
    print(f"  {folder}: {len(pngs)} images")
    total += len(pngs)

print(f"\n✅ Total: {total} images found!")

# === CREATE TEST SPLIT (1,556 images) ===
allimagepaths, alllabels = [], []
for folder in sourcefolders:
    classdir = os.path.join(sourcebase, folder)
    paths = glob(os.path.join(classdir, '*.png'))
    allimagepaths.extend(paths)
    alllabels.extend([folder] * len(paths))

df = pd.DataFrame({'path': allimagepaths, 'label': alllabels})
dftrain, dftemp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
dfval, dftest = train_test_split(dftemp, test_size=0.5, random_state=42, stratify=dftemp['label'])
print(f"✅ Test set ready: {len(dftest)} images")

# === CREATE NEW FOLDER testlightgauss ===
lightgauss_dir = os.path.join(stressbase, 'testlightgauss')
for folder in sourcefolders:
    os.makedirs(os.path.join(lightgauss_dir, folder), exist_ok=True)

print("\n🎯 Creating LIGHT GAUSSIAN (std=10)...")
count = 0
for idx, row in dftest.iterrows():
    imgpath = row['path']
    foldername = row['label']
    img = cv2.imread(imgpath)
    if img is None:
        print(f"⚠️ Skip: {imgpath}")
        continue

    # LIGHT noise std=10 (realistic)
    noise = np.random.normal(0, 10, img.shape).astype(np.uint8)
    light_img = cv2.add(img, noise)

    # SAVE to NEW folder (YOUR exact structure)
    outpath = os.path.join(lightgauss_dir, foldername, os.path.basename(imgpath))
    cv2.imwrite(outpath, light_img)
    count += 1

print(f"\n✅ SUCCESS! Created testlightgauss/ ({count} images)")
print("📁 Location:", lightgauss_dir)
print("🛡️ Original folders → 100% UNTOUCHED")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔍 Verifying YOUR folders:
  Evening_Fire_Incident_aug_img: 0 images
  Evening_Forest_Condition_aug_img: 0 images
  Pre-_Evening_Fire_Incident_aug_img: 0 images
  Pre-_Evening_Forest_Condition_aug_img: 0 images

✅ Total: 0 images found!


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
import os
basepath = '/content/drive/MyDrive/MHCNNFD_Data'
print("📁 ACTUAL FOLDERS in MHCNNFDData/:")
for item in sorted(os.listdir(basepath)):
    fullpath = os.path.join(basepath, item)
    if os.path.isdir(fullpath):
        png_count = len([f for f in os.listdir(fullpath) if f.endswith('.png')])
        print(f"  📂 {item:<40} → {png_count} PNGs")
    else:
        print(f"  📄 {item}")

📁 ACTUAL FOLDERS in MHCNNFDData/:
  📂 Evening_Fire_Incident_aug_img            → 3890 PNGs
  📄 Evening_Fire_Incident_aug_img.zip
  📂 Evening_Forest_Condition_aug_img         → 3890 PNGs
  📄 Evening_Forest_Condition_aug_img.zip
  📂 Pre-_Evening_Fire_Incident_aug_img       → 3890 PNGs
  📄 Pre-_Evening_Fire_Incident_aug_img.zip
  📂 Pre-_Evening_Forest_Condition_aug_img    → 3890 PNGs
  📄 Pre-_Evening_Forest_Condition_aug_img.zip


In [ ]:
# === EXACT STRUCTURE ===
import cv2, numpy as np, os, pandas as pd
from sklearn.model_selection import train_test_split
from glob import glob

sourcebase = '/content/drive/MyDrive/MHCNNFD_Data'
stressbase = '/content/drive/MyDrive/MHCNNFDStressTest'

# ✅ EXACT FOLDER NAMES + 3890 PNGs each
sourcefolders = [
    'Evening_Fire_Incident_aug_img',
    'Evening_Forest_Condition_aug_img',
    'Pre-_Evening_Fire_Incident_aug_img',
    'Pre-_Evening_Forest_Condition_aug_img'
]

print("🔍 Confirmed 15,560 images across 4 classes...")

# === CREATE TEST SPLIT (exactly 1,556 images) ===
allimagepaths, alllabels = [], []
for folder in sourcefolders:
    classdir = os.path.join(sourcebase, folder)
    paths = glob(os.path.join(classdir, '*.png'))
    print(f"  Loaded {len(paths)} from {folder}")
    allimagepaths.extend(paths)
    alllabels.extend([folder] * len(paths))

df = pd.DataFrame({'path': allimagepaths, 'label': alllabels})
print(f"\n✅ Total: {len(df)} images → Splitting 80/10/10...")

dftrain, dftemp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
dfval, dftest = train_test_split(dftemp, test_size=0.5, random_state=42, stratify=dftemp['label'])
print(f"✅ Test set: {len(dftest)} images (389 per class)")

# === CREATE NEW FOLDER testlightgauss (std=10) ===
lightgauss_dir = os.path.join(stressbase, 'testlightgauss')
for folder in sourcefolders:
    os.makedirs(os.path.join(lightgauss_dir, folder), exist_ok=True)

print("\n🎯 Creating LIGHT GAUSSIAN std=10 (realistic drone noise)...")
count = 0
for idx, row in dftest.iterrows():
    imgpath = row['path']
    foldername = row['label']
    img = cv2.imread(imgpath)

    if img is not None:
        # Add realistic light noise
        noise = np.random.normal(0, 10, img.shape).astype(np.uint8)
        light_img = cv2.add(img, noise)

        # Write to NEW folder only
        outpath = os.path.join(lightgauss_dir, foldername, os.path.basename(imgpath))
        cv2.imwrite(outpath, light_img)
        count += 1

print(f"\n✅ SUCCESS! testlightgauss/ created ({count} images)")
print("📁 Full path:", lightgauss_dir)
print("🛡️ MHCNNFD_Data/ → 100% UNTOUCHED")
print("🛡️ Existing testgaussian/, testblur/, testfog/ → 100% UNTOUCHED")


🔍 Confirmed 15,560 images across 4 classes...
  Loaded 3890 from Evening_Fire_Incident_aug_img
  Loaded 3890 from Evening_Forest_Condition_aug_img
  Loaded 3890 from Pre-_Evening_Fire_Incident_aug_img
  Loaded 3890 from Pre-_Evening_Forest_Condition_aug_img

✅ Total: 15560 images → Splitting 80/10/10...
✅ Test set: 1556 images (389 per class)

🎯 Creating LIGHT GAUSSIAN std=10 (realistic drone noise)...

✅ SUCCESS! testlightgauss/ created (1556 images)
📁 Full path: /content/drive/MyDrive/MHCNNFDStressTest/testlightgauss
🛡️ MHCNNFD_Data/ → 100% UNTOUCHED
🛡️ Existing testgaussian/, testblur/, testfog/ → 100% UNTOUCHED


In [ ]:
import os

stressbase = '/content/drive/MyDrive/MHCNNFDStressTest'
lightgauss_path = f"{stressbase}/testlightgauss"

print("📊 Counting images in testlightgauss...")
print("-" * 50)

if os.path.exists(lightgauss_path):
    total_images = 0
    class_counts = {}

    for class_folder in os.listdir(lightgauss_path):
        class_path = os.path.join(lightgauss_path, class_folder)
        if os.path.isdir(class_path):
            png_count = len([f for f in os.listdir(class_path) if f.endswith('.png')])
            class_counts[class_folder] = png_count
            total_images += png_count
            print(f"  {class_folder:<30}: {png_count} images")

    print("-" * 50)
    print(f"  TOTAL IMAGES: {total_images}")
    print(f"  EXPECTED: 1556 (389 × 4 classes)")

else:
    print("❌ Folder not found. Check your stressbase path.")


📊 Counting images in testlightgauss...
--------------------------------------------------
  Evening_Fire_Incident_aug_img : 389 images
  Evening_Forest_Condition_aug_img: 389 images
  Pre-_Evening_Fire_Incident_aug_img: 389 images
  Pre-_Evening_Forest_Condition_aug_img: 389 images
--------------------------------------------------
  TOTAL IMAGES: 1556
  EXPECTED: 1556 (389 × 4 classes)
